# Microbiome & IBD Study — Step 2: Preprocessing

Based on the EDA findings, we need to address:

| Issue | Solution |
|---|---|
| `fecalcal` has 60% NaN | Drop as feature |
| 440 bacteria with >90% zeros | Remove low-prevalence features |
| Abundances are right-skewed | CLR (Centered Log-Ratio) transformation |
| Class imbalance (CD 47%, UC 27%, H 26%) | Compare SMOTE vs class_weight |
| Features on different scales | StandardScaler after CLR |

Following the literature:
- **Su et al. (2022)** use `class_weight='balanced'` and a 70/30 split
- **Freitas et al. (2023)** use SVM-SMOTE with an 85/15 split
- **Manandhar et al. (2021)** use a 70/30 split without oversampling

We will **experimentally compare** both imbalance strategies and both split ratios, and select the best combination in the modelling notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/Microbiome_Metagenomics.csv')
print(f'Dataset shape: {df.shape}')

## 2.1 Separate Metadata and Features

In [ ]:
meta_cols = ['External ID', 'Participant ID', 'week_num', 'diagnosis', 'fecalcal']
bacteria_cols = [c for c in df.columns if c not in meta_cols]

print(f'Bacterial species (original): {len(bacteria_cols)}')
print(f'Metadata columns            : {meta_cols}')

## 2.2 Drop Low-Prevalence Features

**Justification:** 440 out of 566 bacteria are zero in >90% of samples. These carry almost no discriminative information and add noise. We keep only bacteria present (>0) in at least 10% of samples, a stricter threshold than Su et al. (2022) who use 5%, but justified by our higher sparsity.

In [ ]:
prevalence = (df[bacteria_cols] > 0).mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(prevalence.values, bins=50, color='#5b8db8', edgecolor='white')
ax.axvline(0.10, color='red', linestyle='--', linewidth=2, label='10% threshold (ours)')
ax.axvline(0.05, color='orange', linestyle='--', linewidth=2, label='5% threshold (Su et al. 2022)')
ax.set_xlabel('Prevalence (fraction of samples where species > 0)')
ax.set_ylabel('Number of species')
ax.set_title('Bacterial Species Prevalence Distribution', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/07_prevalence.png', bbox_inches='tight')
plt.show()

keep_bacteria = prevalence[prevalence >= 0.10].index.tolist()
print(f'Species kept (>=10% prevalence): {len(keep_bacteria)}')
print(f'Species removed               : {len(bacteria_cols) - len(keep_bacteria)}')

## 2.3 CLR Transformation

**Justification:** Microbiome relative abundance data is **compositional** — values sum to 1 per sample, creating spurious correlations. The **Centered Log-Ratio (CLR)** transformation is the standard solution (Aitchison, 1986). Unlike Su et al. (2022) and Manandhar et al. (2021) who use raw relative abundances, we apply CLR to correct compositionality and reduce right-skew, which should improve model performance.

Formula: `CLR(x_i) = log(x_i / geometric_mean(x))`

In [ ]:
def clr_transform(X, pseudocount=1e-6):
    """Centered Log-Ratio transformation for compositional microbiome data."""
    X_pseudo = X + pseudocount
    log_X = np.log(X_pseudo)
    geometric_mean = log_X.mean(axis=1, keepdims=True)
    return log_X - geometric_mean

X_raw = df[keep_bacteria].values
X_clr = clr_transform(X_raw)

# Compare before/after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(X_raw[:, 0], bins=50, color='#e07b54', edgecolor='white')
axes[0].set_title('Before CLR — right-skewed', fontweight='bold')
axes[0].set_xlabel('Relative abundance')

axes[1].hist(X_clr[:, 0], bins=50, color='#6aab6a', edgecolor='white')
axes[1].set_title('After CLR — more symmetric', fontweight='bold')
axes[1].set_xlabel('CLR value')

plt.suptitle('Effect of CLR Transformation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/08_clr_transformation.png', bbox_inches='tight')
plt.show()

## 2.4 Encode Target Variable

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['diagnosis'])

print('Label encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

## 2.5 Experiment A — Comparing Train/Test Split Ratios

The literature uses different splits:
- Su et al. (2022): **70/30**
- Freitas et al. (2023): **85/15**
- Our baseline: **80/20**

We compare all three using stratified splits. The best will be selected based on model performance in the modelling notebook.

**Key principle:** all splits are done **before** any oversampling to avoid data leakage.

In [ ]:
splits = {
    '70/30': 0.30,
    '80/20': 0.20,
    '85/15': 0.15
}

split_results = {}

for name, test_size in splits.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_clr, y, test_size=test_size, random_state=42, stratify=y
    )
    split_results[name] = {
        'X_train': X_tr, 'X_test': X_te,
        'y_train': y_tr, 'y_test': y_te
    }
    print(f'Split {name}: train={X_tr.shape[0]}, test={X_te.shape[0]}')

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#e07b54', '#5b8db8', '#6aab6a']

for ax, (name, data) in zip(axes, split_results.items()):
    train_counts = pd.Series(data['y_train']).map(
        {i: c for i, c in enumerate(le.classes_)}).value_counts()
    test_counts  = pd.Series(data['y_test']).map(
        {i: c for i, c in enumerate(le.classes_)}).value_counts()

    x = np.arange(len(train_counts))
    ax.bar(x - 0.2, train_counts.values, 0.35, label='Train', color='#5b8db8', edgecolor='white')
    ax.bar(x + 0.2, test_counts.values,  0.35, label='Test',  color='#e07b54', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels([c[:6] for c in train_counts.index], rotation=15)
    ax.set_title(f'Split {name}', fontweight='bold')
    ax.legend()
    ax.set_ylabel('Samples')

plt.suptitle('Class Distribution Across Split Ratios', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/09_split_comparison.png', bbox_inches='tight')
plt.show()

## 2.6 Experiment B — Comparing Imbalance Strategies

We compare two approaches from the literature:

| Strategy | How it works | Used by |
|---|---|---|
| **SMOTE** | Creates synthetic minority samples by interpolating between neighbours | Freitas et al. (2023) |
| **class_weight='balanced'** | Penalises misclassification of minority classes more during training | Su et al. (2022) |

We apply SMOTE here in preprocessing. `class_weight` is a model parameter set directly during training — so both strategies will be tested in the modelling notebook.

In [ ]:
# We use the 80/20 split as our reference for the SMOTE comparison
X_train_raw = split_results['80/20']['X_train']
y_train_raw = split_results['80/20']['y_train']
X_test       = split_results['80/20']['X_test']
y_test       = split_results['80/20']['y_test']

# Scale BEFORE SMOTE (fit on train only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test)

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train_raw)

# Visualise SMOTE effect
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
before = pd.Series(y_train_raw).map({i: c for i, c in enumerate(le.classes_)}).value_counts()
after  = pd.Series(y_train_smote).map({i: c for i, c in enumerate(le.classes_)}).value_counts()
bar_colors = ['#e07b54', '#6aab6a', '#5b8db8']

axes[0].bar(before.index, before.values, color=bar_colors, edgecolor='white')
axes[0].set_title('Strategy 1: SMOTE\n(synthetic samples added)', fontweight='bold')
axes[0].set_ylabel('Samples')
for i, v in enumerate(before.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

axes[1].bar(after.index, after.values, color=bar_colors, edgecolor='white')
axes[1].set_title('After SMOTE\n(classes balanced)', fontweight='bold')
for i, v in enumerate(after.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.suptitle('SMOTE vs Original Training Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/10_smote_effect.png', bbox_inches='tight')
plt.show()

print(f'Train samples before SMOTE: {X_train_scaled.shape[0]}')
print(f'Train samples after  SMOTE: {X_train_smote.shape[0]}')
print()
print('Strategy 2: class_weight="balanced" — no preprocessing needed.')
print('This will be passed directly as a model parameter in the modelling notebook.')

## 2.7 Save All Preprocessed Variants

We save all split variants + both imbalance versions so the modelling notebook can test all combinations.

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Save all split variants (scaled, no SMOTE — class_weight version)
for name, data in split_results.items():
    key = name.replace('/', '_')
    scaler_tmp = StandardScaler()
    X_tr_sc = scaler_tmp.fit_transform(data['X_train'])
    X_te_sc = scaler_tmp.transform(data['X_test'])

    np.save(f'../data/processed/X_train_{key}.npy', X_tr_sc)
    np.save(f'../data/processed/X_test_{key}.npy',  X_te_sc)
    np.save(f'../data/processed/y_train_{key}.npy', data['y_train'])
    np.save(f'../data/processed/y_test_{key}.npy',  data['y_test'])
    print(f'Saved split {name}: train={X_tr_sc.shape[0]}, test={X_te_sc.shape[0]}')

# Save SMOTE version (80/20)
np.save('../data/processed/X_train_80_20_smote.npy', X_train_smote)
np.save('../data/processed/X_test_80_20.npy',        X_test_scaled)
np.save('../data/processed/y_train_80_20_smote.npy', y_train_smote)
np.save('../data/processed/y_test_80_20.npy',        y_test)

# Save feature names and class names
pd.Series(keep_bacteria).to_csv('../data/processed/feature_names.csv', index=False)
pd.Series(le.classes_).to_csv('../data/processed/class_names.csv', index=False)

print(f'\nFeatures saved: {len(keep_bacteria)}')
print(f'Classes       : {list(le.classes_)}')

## 2.8 Preprocessing Summary

| Step | Action | Justification |
|---|---|---|
| Drop `fecalcal` | Removed from features | 60% missing values |
| Prevalence filter | Kept species present in ≥10% samples | Removes noise from ultra-rare taxa (cf. Su et al. 2022) |
| CLR transform | `log(x / geometric_mean(x))` | Corrects compositionality and skewness (Aitchison, 1986) |
| Train/test split | Compared 70/30, 80/20, 85/15 — all stratified | Best ratio selected empirically in modelling notebook |
| StandardScaler | Fit on train only | Prevents data leakage |
| Imbalance | Compared SMOTE vs class_weight | Best strategy selected empirically in modelling notebook |

### Decision to be made in Notebook 3
We will train quick baseline models with each combination and select the **split + imbalance strategy** that yields the best **macro F1-score** on the test set — an imbalance-aware metric.